# Methode - SMOTE pour ameliorer la classe Entree


In [1]:
!pip install -q scikit-learn imbalanced-learn pandas nltk

In [2]:
import pandas as pd
import numpy as np
import re
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import f1_score, classification_report
from imblearn.over_sampling import SMOTE
from google.colab import files

print("Imports OK")

Imports OK


In [4]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

print(f"Train : {len(train)} | Test : {len(test)}")
print(train["type"].value_counts())

Train : 12473 | Test : 1388
type
Plat principal    5802
Dessert           3762
Entrée            2909
Name: count, dtype: int64


In [5]:
stemmer = SnowballStemmer("french")
stop_fr = set(stopwords.words("french"))

def preprocess(text):
    text = re.sub(r"[^\w\s]", " ", str(text))
    text = re.sub(r"\d+", "", text)
    tokens = text.lower().split()
    tokens = [t for t in tokens if t not in stop_fr and len(t) > 2]
    tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)

x_train_text = train[["titre", "ingredients", "recette"]].fillna("").apply(lambda r: " ".join(r), axis=1).apply(preprocess)
x_test_text  = test[["titre", "ingredients", "recette"]].fillna("").apply(lambda r: " ".join(r), axis=1).apply(preprocess)

y_train = train["type"]
y_test  = test["type"]

print("Preprocessing OK")

Preprocessing OK


In [6]:
# TF-IDF
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2), sublinear_tf=True, min_df=2)
x_train_tfidf = tfidf.fit_transform(x_train_text)
x_test_tfidf  = tfidf.transform(x_test_text)

print(f"TF-IDF shape train : {x_train_tfidf.shape}")

TF-IDF shape train : (12473, 10000)


In [7]:
# Baseline sans SMOTE
clf_base = SVC(kernel="linear", C=1.0, random_state=42)
clf_base.fit(x_train_tfidf, y_train)
y_pred_base = clf_base.predict(x_test_tfidf)

f1_base = f1_score(y_test, y_pred_base, average="macro")

print("=== BASELINE - TF-IDF + SVM ===")
print(classification_report(y_test, y_pred_base))
print(f"F1 macro : {f1_base:.3f}")

=== BASELINE - TF-IDF + SVM ===
                precision    recall  f1-score   support

       Dessert       0.99      1.00      0.99       407
        Entrée       0.75      0.71      0.73       337
Plat principal       0.85      0.88      0.87       644

      accuracy                           0.87      1388
     macro avg       0.87      0.86      0.86      1388
  weighted avg       0.87      0.87      0.87      1388

F1 macro : 0.863


In [8]:
# SMOTE - rééchantillonnage de la classe Entree
print(f"Distribution avant SMOTE : {dict(y_train.value_counts())}")

smote = SMOTE(random_state=42)
x_train_smote, y_train_smote = smote.fit_resample(x_train_tfidf, y_train)

print(f"Distribution apres SMOTE : {dict(pd.Series(y_train_smote).value_counts())}")

Distribution avant SMOTE : {'Plat principal': np.int64(5802), 'Dessert': np.int64(3762), 'Entrée': np.int64(2909)}
Distribution apres SMOTE : {'Plat principal': np.int64(5802), 'Entrée': np.int64(5802), 'Dessert': np.int64(5802)}


In [9]:
clf_smote = SVC(kernel="linear", C=1.0, random_state=42)
clf_smote.fit(x_train_smote, y_train_smote)
y_pred_smote = clf_smote.predict(x_test_tfidf)

f1_smote = f1_score(y_test, y_pred_smote, average="macro")

print("===  TF-IDF + SMOTE ===")
print(classification_report(y_test, y_pred_smote))
print(f"F1 macro : {f1_smote:.3f}")

===  TF-IDF + SMOTE ===
                precision    recall  f1-score   support

       Dessert       0.99      1.00      0.99       407
        Entrée       0.71      0.77      0.74       337
Plat principal       0.88      0.84      0.86       644

      accuracy                           0.87      1388
     macro avg       0.86      0.87      0.86      1388
  weighted avg       0.87      0.87      0.87      1388

F1 macro : 0.865


In [10]:
print("=" * 55)
print("RECAP - Impact SMOTE sur la classe Entree")
print("=" * 55)
print(f"{'Methode':<30} F1 macro   F1 Entree")
print("-" * 55)

def get_f1_entree(y_true, y_pred):
    report = classification_report(y_true, y_pred, output_dict=True)
    for key in report:
        if "ntr" in key:
            return report[key]["f1-score"]
    return 0.0

print(f"{'Baseline TF-IDF + SVM':<30} {f1_base:.3f}      {get_f1_entree(y_test, y_pred_base):.3f}")
print(f"{'TF-IDF + SMOTE':<30} {f1_smote:.3f}      {get_f1_entree(y_test, y_pred_smote):.3f}")

RECAP - Impact SMOTE sur la classe Entree
Methode                        F1 macro   F1 Entree
-------------------------------------------------------
Baseline TF-IDF + SVM          0.863      0.729
TF-IDF + SMOTE                 0.865      0.742
